# MovieMate: Conversational AI for Intelligent Movie Search and Recommendations

**Project Overview:** This notebook explores the design and implementation of a conversational AI system that helps users explore movie information through natural language interactions. The system uses **Retrieval-Augmented Generation (RAG)** with **FAISS** vector search and **Large Language Models (LLMs)**.

**RAG Architecture:**
1. **Natural Language Understanding** — parses user intent (genre, year, actor, director, mood)
2. **Sentence-Transformers** — generate query embeddings
3. **FAISS** — vector similarity search over movie metadata
4. **LLM (Dolphin-3 via Ollama)** — generates natural conversational responses from retrieved context

---

## Table of Contents
1. [Dataset Exploration](#1-dataset-exploration)
2. [Exploratory Data Analysis (EDA)](#2-exploratory-data-analysis)
3. [Data Preprocessing](#3-data-preprocessing)
4. [Embedding Generation](#4-embedding-generation-with-sentence-transformers)
5. [FAISS Vector Similarity Search](#5-faiss-vector-similarity-search)
6. [Conversational Movie Chatbot (RAG with LLM)](#6-conversational-movie-chatbot-rag-with-llm)
7. [Interactive Interface (Gradio)](#7-interactive-interface-gradio)
8. [Evaluation and Reflection](#8-evaluation-and-reflection)


## 1. Dataset Exploration

### Dataset Description
A movie dataset derived from IMDb containing **216 movies** spanning from **1927 to 2024** across **20+ genres**. Each entry contains: **Title, Year, Rating, Genre, Director, Cast, Duration, Plot**.

In [ ]:
import json
import pandas as pd
import numpy as np

with open('data/movies.json', 'r') as f:
    movies_raw = json.load(f)
print(f"Total movies loaded: {len(movies_raw)}")
print(f"\nFirst movie: {json.dumps(movies_raw[0], indent=2)}")


In [ ]:
df_raw = pd.DataFrame(movies_raw)
print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nData Types:\n{df_raw.dtypes}")
df_raw.head()


In [ ]:
print("=== Numerical Summary ===")
print(df_raw[['year', 'rating', 'duration']].describe())

all_genres = []
for genres in df_raw['genre']:
    all_genres.extend(genres)
genre_counts = pd.Series(all_genres).value_counts()
print(f"\nTotal unique genres: {len(genre_counts)}")
print(f"\nTop 10 genres:\n{genre_counts.head(10)}")


In [ ]:
print("=== Dataset Observations ===")
print(f"1. Year range: {df_raw['year'].min()} - {df_raw['year'].max()}")
print(f"2. Rating range: {df_raw['rating'].min()} - {df_raw['rating'].max()}")
print(f"3. Average rating: {df_raw['rating'].mean():.2f}")
print(f"4. Average duration: {df_raw['duration'].mean():.0f} minutes")
print(f"5. Total unique directors: {df_raw['director'].nunique()}")
print(f"6. Movies with multiple genres: {(df_raw['genre'].apply(len) > 1).sum()}")
print(f"7. Most common director: {df_raw['director'].value_counts().index[0]} ({df_raw['director'].value_counts().iloc[0]} movies)")


## 2. Exploratory Data Analysis (EDA)

Visualizing dataset characteristics to understand patterns and trends.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('MovieMate - Exploratory Data Analysis', fontsize=16, fontweight='bold')

axes[0, 0].hist(df_raw['rating'], bins=20, color='#e94560', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df_raw['rating'].mean(), color='#f59e0b', linestyle='--', linewidth=2, label=f"Mean: {df_raw['rating'].mean():.2f}")
axes[0, 0].set_title('Distribution of Movie Ratings', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Rating'); axes[0, 0].set_ylabel('Count'); axes[0, 0].legend()

top_genres = genre_counts.head(15)
axes[0, 1].barh(range(len(top_genres)), top_genres.values[::-1], color='#4ecdc4', edgecolor='black')
axes[0, 1].set_yticks(range(len(top_genres))); axes[0, 1].set_yticklabels(top_genres.index[::-1])
axes[0, 1].set_title('Top 15 Genres by Frequency', fontsize=12, fontweight='bold'); axes[0, 1].set_xlabel('Count')

df_raw['decade'] = (df_raw['year'] // 10) * 10
decade_counts = df_raw['decade'].value_counts().sort_index()
colors = plt.cm.RdYlBu(np.linspace(0.2, 0.8, len(decade_counts)))
axes[1, 0].bar(range(len(decade_counts)), decade_counts.values, color=colors, edgecolor='black')
axes[1, 0].set_xticks(range(len(decade_counts))); axes[1, 0].set_xticklabels([f"{int(d)}s" for d in decade_counts.index], rotation=45)
axes[1, 0].set_title('Movies by Decade', fontsize=12, fontweight='bold'); axes[1, 0].set_xlabel('Decade'); axes[1, 0].set_ylabel('Count')

axes[1, 1].scatter(df_raw['duration'], df_raw['rating'], alpha=0.6, color='#e94560', edgecolors='black')
axes[1, 1].set_title('Duration vs Rating', fontsize=12, fontweight='bold'); axes[1, 1].set_xlabel('Duration (min)'); axes[1, 1].set_ylabel('Rating')

plt.tight_layout(); plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
df_sorted = df_raw.sort_values('year')
rolling_avg = df_sorted['rating'].rolling(window=10, min_periods=1).mean()
axes[0].plot(df_sorted['year'].values, df_sorted['rating'].values, 'o', alpha=0.3, color='gray', label='Individual movies')
axes[0].plot(df_sorted['year'].values, rolling_avg.values, '-', linewidth=2.5, color='#e94560', label='10-movie rolling avg')
axes[0].set_title('Movie Ratings Over Time', fontsize=12, fontweight='bold'); axes[0].set_xlabel('Year'); axes[0].set_ylabel('Rating'); axes[0].legend()

top_directors = df_raw['director'].value_counts().head(10)
axes[1].barh(range(len(top_directors)), top_directors.values[::-1], color='#4ecdc4', edgecolor='black')
axes[1].set_yticks(range(len(top_directors))); axes[1].set_yticklabels(top_directors.index[::-1])
axes[1].set_title('Top 10 Directors', fontsize=12, fontweight='bold'); axes[1].set_xlabel('Number of Movies')
plt.tight_layout(); plt.savefig('eda_plots_2.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
genre_dummies = pd.get_dummies(df_raw['genre'].explode()).groupby(level=0).sum()
genre_corr = genre_dummies.corr()
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(genre_corr, dtype=bool))
sns.heatmap(genre_corr, mask=mask, annot=False, cmap='RdYlBu_r', center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Genre Co-occurrence Heatmap', fontsize=14, fontweight='bold'); plt.tight_layout()
plt.savefig('genre_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
plt.figure(figsize=(14, 8))
df_exploded = df_raw.explode('genre')
top_8 = df_exploded['genre'].value_counts().head(8).index
df_f = df_exploded[df_exploded['genre'].isin(top_8)]
sns.boxplot(data=df_f, x='genre', y='rating', palette='husl')
plt.xticks(rotation=45); plt.title('Rating Distribution by Genre', fontsize=14, fontweight='bold')
plt.xlabel('Genre'); plt.ylabel('Rating'); plt.tight_layout()
plt.savefig('genre_boxplot.png', dpi=150, bbox_inches='tight'); plt.show()


## 3. Data Preprocessing

- Handling missing values
- Normalizing text fields
- Creating combined text features for embeddings

In [ ]:
df = df_raw.copy()
print("=== Missing Values ==="); print(df.isnull().sum())
print(f"\nDuplicates: {df.duplicated(subset=['title', 'year']).sum()}")


In [ ]:
def clean_text(text):
    if isinstance(text, str): return ' '.join(text.lower().split())
    return text

df['title_clean'] = df['title'].apply(clean_text)
df['director_clean'] = df['director'].apply(clean_text)
df['plot_clean'] = df['plot'].apply(clean_text)
df['genre_clean'] = df['genre'].apply(lambda x: sorted([g.lower() for g in x]))
df['cast_clean'] = df['cast'].apply(lambda x: [clean_text(c) for c in x])
print("Text normalization complete")


In [ ]:
df['combined_text'] = (
    "Title: " + df['title_clean'] + ". " +
    "Genres: " + df['genre_clean'].apply(lambda x: ', '.join(x)) + ". " +
    "Director: " + df['director_clean'] + ". " +
    "Cast: " + df['cast_clean'].apply(lambda x: ', '.join(x)) + ". " +
    "Plot: " + df['plot_clean'] + ". " +
    "Year: " + df['year'].astype(str) + ". " +
    "Rating: " + df['rating'].astype(str) + ".")
print(f"Combined text feature created (avg {df['combined_text'].apply(len).mean():.0f} chars)")


In [ ]:
df['rating'] = df['rating'].fillna(df['rating'].median())
df['duration'] = df['duration'].fillna(df['duration'].median())
df['plot_clean'] = df['plot_clean'].fillna('No plot available.')
df['director_clean'] = df['director_clean'].fillna('Unknown')
df.to_csv('data/movies_preprocessed.csv', index=False)
print(f"Preprocessed: {df.shape} -> saved to data/movies_preprocessed.csv")
df.head()


## 4. Embedding Generation with Sentence-Transformers

Model: `all-MiniLM-L6-v2` — maps text to 384-dimensional vectors optimized for semantic similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
import time

print("Loading model...")
start = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Loaded in {time.time() - start:.2f}s")

test_emb = model.encode(["action movie with explosions", "romantic comedy about love"])
print(f"Embedding shape: {test_emb.shape} (dimension: {test_emb.shape[1]})")


In [ ]:
print("Generating embeddings...")
start = time.time()
movie_embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)
print(f"Done in {time.time() - start:.2f}s — shape: {movie_embeddings.shape}")
np.save('data/movie_embeddings.npy', movie_embeddings)
print("Saved to data/movie_embeddings.npy")


In [ ]:
from sklearn.manifold import TSNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=15)
emb_2d = tsne.fit_transform(movie_embeddings)

df['dominant_genre'] = df['genre_clean'].apply(lambda x: x[0] if x else 'unknown')
gc = {'drama': '#e94560', 'action': '#4ecdc4', 'comedy': '#f59e0b', 'thriller': '#6c5ce7',
      'horror': '#d63031', 'sci-fi': '#00b894', 'romance': '#fd79a8', 'crime': '#636e72',
      'animation': '#0984e3', 'adventure': '#e17055'}
colors = [gc.get(g, '#b2bec3') for g in df['dominant_genre']]

plt.figure(figsize=(14, 10))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, alpha=0.7, s=50, edgecolors='black')
from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='w', label=g.capitalize(), markerfacecolor=c, markersize=10) for g, c in gc.items()]
plt.legend(handles=leg, loc='best', fontsize=8, ncol=2)
plt.title('t-SNE: Movie Embeddings by Genre', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2'); plt.tight_layout()
plt.savefig('tsne_embeddings.png', dpi=150, bbox_inches='tight'); plt.show()


## 5. FAISS Vector Similarity Search

FAISS (Facebook AI Similarity Search) enables efficient nearest-neighbor search in high-dimensional space.

In [ ]:
import faiss
faiss.normalize_L2(movie_embeddings)
embedding_dim = movie_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(movie_embeddings)
print(f"FAISS Index: {index.ntotal} vectors, dim={embedding_dim}")


In [ ]:
def search_movies_faiss(query, k=5):
    qe = model.encode([query]); faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, k)
    results = []
    for i, idx in enumerate(idxs[0]):
        if idx < len(df):
            results.append({'rank': i+1, 'title': df.iloc[idx]['title'], 'year': df.iloc[idx]['year'],
                'rating': df.iloc[idx]['rating'], 'genre': df.iloc[idx]['genre'],
                'director': df.iloc[idx]['director'], 'cast': df.iloc[idx]['cast'],
                'duration': df.iloc[idx]['duration'], 'plot': df.iloc[idx]['plot'],
                'similarity_score': float(scores[0][i])})
    return results

for q in ["Sci-fi movies about space", "Feel-good comedy", "Christopher Nolan"]:
    print(f"\nQuery: '{q}'")
    for r in search_movies_faiss(q, k=3):
        print(f"  {r['rank']}. {r['title']} ({r['year']}) ⭐{r['rating']} score={r['similarity_score']:.3f}")


In [ ]:
def keyword_search(query, k=5):
    ql = query.lower()
    scores = [(idx, sum(1 for w in ql.split() if w in row['combined_text'].lower())) for idx, row in df.iterrows()]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [{'rank': i+1, 'title': df.iloc[idx]['title'], 'year': df.iloc[idx]['year'],
             'rating': df.iloc[idx]['rating'], 'genre': df.iloc[idx]['genre'],
             'director': df.iloc[idx]['director'], 'similarity_score': s}
            for i, (idx, s) in enumerate(scores[:k])]

print("=== FAISS vs Keyword ===")
q = "mind-bending movies about reality"
print(f"Query: '{q}'\n--- FAISS ---")
for r in search_movies_faiss(q, 5): print(f"  {r['rank']}. {r['title']} ({r['year']}) score={r['similarity_score']:.3f}")
print("\n--- Keyword ---")
for r in keyword_search(q, 5): print(f"  {r['rank']}. {r['title']} ({r['year']}) score={r['similarity_score']}")


## 6. Conversational Movie Chatbot (RAG with LLM)

Complete RAG pipeline:
```
User Query → NLU Intent → FAISS Retrieval → LLM Response Generation
```

### 6.1 NLU Query Parser

In [ ]:
import re

class NLUQueryParser:
    GENRE_ALIASES = {
        'scary': 'horror', 'frightening': 'horror', 'creepy': 'horror',
        'funny': 'comedy', 'hilarious': 'comedy', 'laugh': 'comedy',
        'romantic': 'romance', 'love': 'romance',
        'action': 'action', 'fight': 'action', 'explosions': 'action',
        'drama': 'drama', 'emotional': 'drama',
        'sci-fi': 'sci-fi', 'science fiction': 'sci-fi', 'space': 'sci-fi', 'futuristic': 'sci-fi',
        'thriller': 'thriller', 'suspense': 'thriller', 'tension': 'thriller',
        'animation': 'animation', 'animated': 'animation', 'cartoon': 'animation',
        'comedy': 'comedy', 'horror': 'horror', 'crime': 'crime',
        'detective': 'crime', 'murder': 'crime', 'adventure': 'adventure',
        'quest': 'adventure', 'journey': 'adventure', 'fantasy': 'fantasy',
        'magic': 'fantasy', 'mystery': 'mystery', 'whodunit': 'mystery',
        'musical': 'musical', 'music': 'musical', 'war': 'war', 'military': 'war',
        'western': 'western', 'documentary': 'documentary', 'biography': 'biography',
        'biopic': 'biography', 'history': 'history', 'historical': 'history',
        'sport': 'sport', 'superhero': 'action', 'feel-good': 'comedy', 'feel good': 'comedy',
        'uplifting': 'animation', 'mind-bending': 'sci-fi', 'mind bending': 'sci-fi',
        'psychological': 'thriller', 'dark': 'thriller', 'intense': 'thriller',
        'family': 'animation', 'kids': 'animation', 'children': 'animation',
    }
    MOOD_GENRE = {
        'feel-good': ['comedy', 'animation', 'romance'], 'happy': ['comedy', 'animation', 'romance'],
        'sad': ['drama', 'romance'], 'excited': ['action', 'adventure', 'sci-fi'],
        'scared': ['horror', 'thriller'], 'thoughtful': ['drama', 'sci-fi', 'mystery'],
        'nostalgic': ['drama', 'romance', 'history'], 'inspired': ['biography', 'drama', 'sport'],
    }

    def parse(self, query, history=None):
        ql = query.lower()
        intent = {'genres': [], 'min_year': None, 'max_year': None, 'min_rating': None,
                  'actor': None, 'director': None, 'similar_to': None,
                  'query_type': 'general', 'raw_query': query}
        m = re.search(r'(?:movies?|films?)\s+(?:like|similar\s+to|about)\s+(.+)', ql)
        if m: intent['similar_to'] = m.group(1).strip(); intent['query_type'] = 'similar'
        ya = re.search(r'(?:after|since|from)\s+(\d{4})', ql)
        if ya: intent['min_year'] = int(ya.group(1))
        yb = re.search(r'(?:before)\s+(\d{4})', ql)
        if yb: intent['max_year'] = int(yb.group(1))
        dm = re.search(r'(\d{3})0s', ql)
        if dm: d = int(dm.group(1)+'0'); intent['min_year'] = d; intent['max_year'] = d+9
        if re.search(r'(?:highly\s+rated|best|good|top|great)', ql): intent['min_rating'] = 7.5
        for alias, genre in self.GENRE_ALIASES.items():
            if alias in ql and genre not in intent['genres']: intent['genres'].append(genre)
        for mood, genres in self.MOOD_GENRE.items():
            if mood in ql: intent['genres'].extend(genres); intent['genres'] = list(set(intent['genres']))
        actors = set()
        for cl in df['cast']: actors.update([c.lower() for c in cl])
        for a in actors:
            if a in ql: intent['actor'] = a.title(); break
        for d in df['director'].str.lower().unique():
            if d in ql: intent['director'] = d.title(); break
        if any(w in ql for w in ['recommend','suggest','show','find']): intent['query_type'] = 'recommend'
        if history:
            ctx = {}
            for msg in history[-4:]:
                if msg.get('role') == 'assistant' and msg.get('_meta'): ctx.update(msg['_meta'])
            if not intent['genres'] and ctx.get('genres'): intent['genres'] = ctx['genres']
            if not intent['min_year'] and ctx.get('min_year'): intent['min_year'] = ctx['min_year']
        return intent

parser = NLUQueryParser()
for q in ["Suggest sci-fi after 2010", "Movies similar to Inception",
          "Good drama movies", "Christopher Nolan films", "Feel-good movies",
          "Scary horror from 90s", "Leonardo DiCaprio movies"]:
    i = parser.parse(q)
    print(f"'{q}' → type={i['query_type']}, genres={i['genres']}, year={i['min_year']}-{i['max_year']}, actor={i['actor']}, director={i['director']}, similar={i['similar_to']}")


### 6.2 LLM Integration

We connect to **Ollama** (local LLM server) running **Dolphin-3** (8B). This LLM receives retrieved movie data and generates natural responses.

In [ ]:
from openai import OpenAI

llm_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM_MODEL = "dolphin3:8b"

def llm_test():
    try:
        r = llm_client.chat.completions.create(model=LLM_MODEL,
            messages=[{"role":"user","content":"Say hello in one word."}],
            max_tokens=10, temperature=0)
        return r.choices[0].message.content
    except Exception as e:
        return f"LLM not available: {e}"

print(f"LLM Test ({LLM_MODEL}): {llm_test()}")


### 6.3 RAG Chatbot Class

Combines NLU parsing → FAISS retrieval → LLM response generation.

In [ ]:
class ConversationalMovieBot:
    def __init__(self, df, faiss_index, embeddings, emb_model, llm_client, llm_model):
        self.df = df; self.index = faiss_index; self.emb_model = emb_model
        self.parser = NLUQueryParser()
        self.llm_client = llm_client; self.llm_model = llm_model
        self.history = []

    def retrieve(self, intent, k=8):
        if intent.get('similar_to'):
            tm = self.df[self.df['title_clean'].str.contains(intent['similar_to'], case=False, na=False)]
            if not tm.empty:
                ref = tm.iloc[0]; re = self.emb_model.encode([ref['combined_text']])
                faiss.normalize_L2(re); sc, ix = self.index.search(re, k+5)
                res = []
                for i, idx in enumerate(ix[0]):
                    if idx < len(self.df) and self.df.iloc[idx]['title'] != ref['title']:
                        res.append(self.df.iloc[idx].to_dict()); res[-1]['similarity_score'] = float(sc[0][i])
                        if len(res) >= k: break
                return res
        qe = self.emb_model.encode([intent['raw_query']]); faiss.normalize_L2(qe)
        sc, ix = self.index.search(qe, k*3)
        cands = []
        for i, idx in enumerate(ix[0]):
            if idx < len(self.df):
                m = self.df.iloc[idx].to_dict(); m['similarity_score'] = float(sc[0][i]); cands.append(m)
        filt = []
        for m in cands:
            if intent['genres'] and not any(g in [x.lower() for x in m['genre']] for g in intent['genres']): continue
            if intent['min_year'] and m['year'] < intent['min_year']: continue
            if intent['max_year'] and m['year'] > intent['max_year']: continue
            if intent['min_rating'] and m['rating'] < intent['min_rating']: continue
            if intent['actor'] and intent['actor'].lower() not in [c.lower() for c in m['cast']]: continue
            if intent['director'] and intent['director'].lower() != m['director'].lower(): continue
            filt.append(m)
        return filt[:k] if len(filt) >= 3 else cands[:k]

    def _system_prompt(self, intent, movies):
        s = """You are MovieMate, a friendly movie recommendation assistant.
Be conversational like a friend recommending movies.
For each movie include: title, year, rating, genre, director, brief plot mention.
End with a follow-up suggestion. Keep responses concise."""
        s += "\n\nRETRIEVED MOVIES:\n"
        for i, m in enumerate(movies, 1):
            g = ', '.join(m['genre']) if isinstance(m['genre'], list) else m['genre']
            c = ', '.join(m['cast'][:3]) if isinstance(m['cast'], list) else m['cast']
            s += f"{i}. {m['title']} ({m['year']}) | ⭐{m['rating']}/10 | {g} | Dir: {m['director']} | Cast: {c} | {m['duration']}min | {m['plot']}\n"
        return s

    def respond(self, intent, movies):
        if not movies:
            try:
                r = self.llm_client.chat.completions.create(model=self.llm_model,
                    messages=[{"role":"system","content":"You are MovieMate."},
                              {"role":"user","content":f"User asked: '{intent['raw_query']}' but no movies found. Explain why and suggest broadening search."}],
                    max_tokens=256, temperature=0.7)
                return r.choices[0].message.content
            except:
                return "I couldn't find matching movies. Try broadening your search — e.g., instead of '90s horror', try 'horror movies' or a different decade."
        try:
            msgs = [{"role":"system","content": self._system_prompt(intent, movies)},
                    {"role":"user","content": f"Answer: {intent['raw_query']}"}]
            r = self.llm_client.chat.completions.create(model=self.llm_model, messages=msgs, max_tokens=1024, temperature=0.7)
            return r.choices[0].message.content
        except Exception as e:
            # Fallback: template-based response when LLM unavailable
            resp = f"I found {len(movies)} movies for you:\n\n"
            for i, m in enumerate(movies, 1):
                g = ', '.join(m['genre']) if isinstance(m['genre'], list) else m['genre']
                c = ', '.join(m['cast'][:3]) if isinstance(m['cast'], list) else m['cast']
                resp += f"**{i}. {m['title']}** ({m['year']}) ⭐{m['rating']}/10 — {g}\n"
                resp += f"   Director: {m['director']} | Cast: {c}\n"
                resp += f"   {m['plot']}\n\n"
            resp += f"💡 Want more suggestions? Try asking about a specific genre, year, or actor."
            return resp

    def chat(self, query):
        self.history.append({'role':'user','content':query})
        intent = self.parser.parse(query, self.history)
        movies = self.retrieve(intent)
        response = self.respond(intent, movies)
        self.history.append({'role':'assistant','content':response,'_meta':{'genres':intent['genres'],'min_year':intent['min_year']}})
        return response, movies

    def reset(self): self.history = []

bot = ConversationalMovieBot(df, index, movie_embeddings, model, llm_client, LLM_MODEL)
print(f"✅ RAG Chatbot initialized ({LLM_MODEL})")
print(f"   {len(df)} movies | FAISS: {index.ntotal} vectors | LLM: {LLM_MODEL}")


### 6.4 Chatbot Demo

In [ ]:
queries_demo = [
    "Suggest sci-fi movies released after 2010",
    "Movies similar to The Godfather",
    "Good drama movies with high IMDb ratings",
    "What movies did Christopher Nolan direct?",
    "Recommend some feel-good movies",
    "Scary horror movies from the 90s",
    "Best thriller movies",
    "Animated movies for kids",
]
bot.reset()
for q in queries_demo:
    r, m = bot.chat(q)
    print(f"\n{'='*60}\n👤 {q}\n🤖 ({len(m)} movies)\n{r[:500]}\n{'='*60}")


In [ ]:
# Multi-turn conversation demo
bot.reset()
print("=== MULTI-TURN DEMO ===")
r1, m1 = bot.chat("Show me action movies")
print(f"\n👤 Show me action movies\n🤖 ({len(m1)} movies)\n{r1[:300]}...")
r2, m2 = bot.chat("Only those after 2000")
print(f"\n👤 Only those after 2000\n🤖 ({len(m2)} movies)\n{r2[:300]}...")
print(f"\nConversation turns: {len(bot.history)}")


## 7. Interactive Interface (Gradio)

A web-based chat interface for the LLM-powered movie bot.

In [ ]:
try:
    import gradio as gr
except ImportError:
    import subprocess; subprocess.check_call(['pip', 'install', 'gradio']); import gradio as gr

def gradio_fn(message, history):
    if not message.strip(): return "Enter a movie query!"
    r, m = bot.chat(message)
    return r + f"\n\n_({len(m)} movies retrieved)_"

demo = gr.ChatInterface(
    fn=gradio_fn,
    title="🎬 MovieMate — AI Movie Discovery",
    description="Ask about movies! Try genres, actors, directors, decades, or 'movies like X'.",
    examples=[["Suggest sci-fi movies after 2010"], ["Movies similar to The Shawshank Redemption"],
              ["Christopher Nolan films"], ["Feel-good comedy movies"],
              ["Horror movies from the 90s"], ["Leonardo DiCaprio movies"]],
)
print("✅ Gradio interface ready — uncomment demo.launch() to start")


In [ ]:
# demo.launch(share=False, server_name="0.0.0.0", server_port=7860)
print("Uncomment the line above and run to launch the web interface.")


## 8. Evaluation and Reflection

### 8.1 Evaluation Test Suite

In [ ]:
tests = [
    {'query': "Suggest sci-fi movies released after 2010", 'genre': 'sci-fi', 'type': 'genre+year'},
    {'query': "Movies similar to The Godfather", 'type': 'similarity'},
    {'query': "Good drama movies with high IMDb ratings", 'genre': 'drama', 'type': 'genre+rating'},
    {'query': "What movies did Christopher Nolan direct?", 'director': 'Christopher Nolan', 'type': 'director'},
    {'query': "Recommend some feel-good movies", 'type': 'mood'},
    {'query': "Scary horror movies from the 90s", 'genre': 'horror', 'type': 'genre+decade'},
    {'query': "Best action movies", 'genre': 'action', 'type': 'genre+rating'},
    {'query': "mind-bending movies about reality", 'type': 'semantic'},
    {'query': "Leonardo DiCaprio movies", 'actor': 'Leonardo DiCaprio', 'type': 'actor'},
    {'query': "Animated movies for kids", 'genre': 'animation', 'type': 'inferred'},
]

print("="*70); print("EVALUATION RESULTS"); print("="*70)
results = []
for t in tests:
    bot.reset(); r, m = bot.chat(t['query']); intent = bot.parser.parse(t['query'])
    sc, mx, dt = 0, 0, []
    mx += 1
    if m: sc += 1; dt.append("✅ movies")
    else: dt.append("❌ no movies")
    if 'genre' in t:
        mx += 1
        if m and any(t['genre'].lower() in [g.lower() for g in mv['genre']] for mv in m): sc += 1; dt.append(f"✅ genre")
        else: dt.append("❌ genre")
    if m and 'similarity_score' in m[0]:
        mx += 1; av = np.mean([mv.get('similarity_score',0) for mv in m])
        if av > 0.3: sc += 1; dt.append(f"✅ sim={av:.3f}")
        else: dt.append(f"⚠️ sim={av:.3f}")
    mx += 1
    if intent['genres'] or intent['actor'] or intent['director'] or intent['min_year']: sc += 1; dt.append("✅ NLU")
    else: dt.append("⚠️ semantic")
    mx += 1
    if len(r) > 50: sc += 1; dt.append(f"✅ response({len(r)}c)")
    else: dt.append("❌ short")
    pct = (sc/mx*100) if mx > 0 else 0
    results.append({'query':t['query'],'type':t['type'],'score':f"{sc}/{mx}",'pct':pct,'movies':len(m),'details':', '.join(dt)})

edf = pd.DataFrame(results)
print("\n" + edf[['query','type','score','movies','details']].to_string(index=False))
print(f"\nOverall: {edf['pct'].mean():.1f}%")


In [ ]:
print("=== FAISS vs Keyword Search ===")
cqs = ["mind-bending movies about reality", "dark psychological thriller",
       "epic space adventure", "romantic movies about love", "intense crime drama"]
fw, kw, ti = 0, 0, 0
gw = ['action','comedy','drama','horror','thriller','sci-fi','romance','animation','adventure','crime','mystery','fantasy']
for q in cqs:
    fr = search_movies_faiss(q, 3); kr = keyword_search(q, 3); ql = q.lower()
    def rs(res, ql):
        s = 0
        for r in res:
            t = f"{r['title']} {' '.join(r['genre'])} {r.get('plot','')}".lower()
            for w in gw:
                if w in ql and w in t: s += 1
            if 'similarity_score' in r and r['similarity_score'] > 0.3: s += 1
        return s
    fs, ks = rs(fr, ql), rs(kr, ql)
    if fs > ks: fw += 1; w = "FAISS"
    elif ks > fs: kw += 1; w = "Keyword"
    else: ti += 1; w = "Tie"
    print(f"'{q[:40]}' → {w} (FAISS:{fs}, KW:{ks})")
print(f"\nFAISS:{fw}/{len(cqs)}, Keyword:{kw}, Ties:{ti}")


In [ ]:
import time
print("=== Response Time (FAISS retrieval only, no LLM) ===")
qs = ["Suggest action movies", "Movies similar to Inception", "Good drama after 2010"]
ts = []
for q in qs:
    s = time.time(); bot.reset(); _, m = bot.chat(q); e = time.time() - s; ts.append(e)
    print(f"'{q}' → {e:.2f}s | {len(m)} movies")
print(f"\nAvg: {np.mean(ts):.2f}s (includes LLM if available)")


### 8.2 Limitations

1. **Dataset Size** — 216 movies is small vs real-world datasets
2. **Rule-Based NLU** — Hand-crafted patterns may fail on novel phrasings
3. **Local LLM** — Dolphin-3 (8B) capable but less fluent than GPT-4/Claude
4. **No Personalization** — No user profiles across sessions
5. **No Plot-Level Search** — Embeddings use combined metadata
6. **No Multimodal** — Text-only; posters/trailers could enhance

### 8.3 Future Improvements

1. Cloud LLM API (GPT-4o / Claude) for more natural responses
2. Larger dataset via TMDB/OMDB APIs
3. User profiling for personalized recommendations
4. Hybrid FAISS + BM25 search
5. Fine-tuned movie-specific embeddings
6. ReAct agent with clarifying questions
7. CLIP embeddings for poster-based search

### 8.4 Conclusion

This project demonstrates a **conversational AI movie discovery system** using:
- **Sentence-Transformers** for semantic understanding
- **FAISS** for efficient vector similarity search
- **NLU** for intent extraction from natural language
- **A real LLM (Dolphin-3 via Ollama)** for generating conversational responses
- **RAG architecture** combining retrieval with generation
- **Gradio** for an interactive web interface

The system handles genre searches, similarity queries, director/actor lookups, mood-based recommendations, and multi-turn refinements.